In [ ]:
#!pip install evaluate

# Vision Transformer for Flower Classification
Author: **Quang-Huy Tran**  
Data: *flower-photos*

In [77]:
import torch
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split
from torchvision import transforms
from transformers import AutoConfig
from transformers import ViTForImageClassification
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

In [46]:
# !unzip -q flower_photos.zip

In [47]:
data = ImageFolder('./flower_photos')
data

Dataset ImageFolder
    Number of datapoints: 3670
    Root location: ./flower_photos

In [48]:
print(f'Flower Data:\nclasses: {data.classes}\nnum samples: {len(data)}')

Flower Data:
classes: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
num samples: 3670


### Data Splitting

In [49]:
train_ratio = 0.8
val_ratio = 0.1

train_size = int(train_ratio*len(data))
val_size = int(val_ratio*len(data))
test_size = len(data) - train_size - val_size

print(f"Data splitting: {train_size, val_size, test_size}")


Data splitting: (2936, 367, 367)


In [50]:
trainset, valset, testset = random_split(data, [train_size, val_size, test_size])

trainset.dataset.samples[:5], '--------------', trainset.dataset.samples[-5:]

([('./flower_photos/daisy/100080576_f52e8ee070_n.jpg', 0),
  ('./flower_photos/daisy/10140303196_b88d3d6cec.jpg', 0),
  ('./flower_photos/daisy/10172379554_b296050f82_n.jpg', 0),
  ('./flower_photos/daisy/10172567486_2748826a8b.jpg', 0),
  ('./flower_photos/daisy/10172636503_21bededa75_n.jpg', 0)],
 '--------------',
 [('./flower_photos/tulips/9831362123_5aac525a99_n.jpg', 4),
  ('./flower_photos/tulips/9870557734_88eb3b9e3b_n.jpg', 4),
  ('./flower_photos/tulips/9947374414_fdf1d0861c_n.jpg', 4),
  ('./flower_photos/tulips/9947385346_3a8cacea02_n.jpg', 4),
  ('./flower_photos/tulips/9976515506_d496c5e72c.jpg', 4)])

### Data Augmentation

In [51]:
img_size = (224, 224)

train_trans = transforms.Compose([
    transforms.Resize(img_size),
    transforms.RandomRotation(0.2),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
  ])

test_trans = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])


trainset.dataset.transform = train_trans
valset.dataset.transform = test_trans
testset.dataset.transform = test_trans

### Fine-tuning model
- Model name: *vit-base-patch16-224-in21k*
- Với dữ liệu dạng ảnh, nên truyền các tham số ánh xạ từ class sang số, và từ số sang class vào model, nếu không model sẽ tạo tự động và không đúng mong đợi của ta.

In [59]:
label2idx = data.class_to_idx
idx2label = {v: k for k, v in label2idx.items()}

label2idx, idx2label

({'daisy': 0, 'dandelion': 1, 'roses': 2, 'sunflowers': 3, 'tulips': 4},
 {0: 'daisy', 1: 'dandelion', 2: 'roses', 3: 'sunflowers', 4: 'tulips'})

In [73]:
model_name = 'google/vit-base-patch16-224-in21k'

cfgs = AutoConfig.from_pretrained(
    model_name,
    num_labels = len(label2idx),
    label2id=label2idx,
    id2label=idx2label,
    finetuning_task = 'image-classification',
)

model = ViTForImageClassification.from_pretrained(
    model_name,
    config=cfgs
)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Training Argument, Trainer

In [80]:
training_args = TrainingArguments(
    output_dir='./train_results',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    num_train_epochs=10,
    load_best_model_at_end=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    metric_for_best_model='accuracy',
    logging_dir='./logs',
    remove_unused_columns=False
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [81]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

def collate_fn(examples):
    # example => Tuple(image, label)
    pixel_values = torch.stack([example[0] for example in examples])
    labels = torch.tensor([example[1] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator = collate_fn,
    train_dataset = trainset,
    eval_dataset = valset,
    compute_metrics = compute_metrics,
)

trainer


In [82]:
import wandb
wandb.init(mode='disabled')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


In [83]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.526957,0.970027
2,No log,0.250326,0.972752
3,No log,0.196879,0.964578
4,No log,0.173298,0.970027
5,No log,0.163176,0.967302
6,0.322839,0.162425,0.961853
7,0.322839,0.154464,0.967302
8,0.322839,0.153507,0.961853
9,0.322839,0.152363,0.961853
10,0.322839,0.151730,0.964578


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=920, training_loss=0.2023500566897185, metrics={'train_runtime': 544.197, 'train_samples_per_second': 53.951, 'train_steps_per_second': 1.691, 'total_flos': 2.2752259898322125e+18, 'train_loss': 0.2023500566897185, 'epoch': 10.0})

In [84]:
trainer.evaluate(testset)

{'eval_loss': 0.23594766855239868,
 'eval_accuracy': 0.9836512261580381,
 'eval_runtime': 2.9312,
 'eval_samples_per_second': 125.206,
 'eval_steps_per_second': 4.094,
 'epoch': 10.0}

In [89]:
repo_id = 'huytqvn/vit-demo'

model.push_to_hub(repo_id, commit_message='tracking best model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...9_tie_q/model.safetensors:   0%|          |  552kB /  343MB            

CommitInfo(commit_url='https://huggingface.co/huytqvn/vit-demo/commit/87e34aa10fb0fd8f5656d900b747a71873ce4c75', commit_message='tracking best model', commit_description='', oid='87e34aa10fb0fd8f5656d900b747a71873ce4c75', pr_url=None, repo_url=RepoUrl('https://huggingface.co/huytqvn/vit-demo', endpoint='https://huggingface.co', repo_type='model', repo_id='huytqvn/vit-demo'), pr_revision=None, pr_num=None)